# Labeling Manual Data Tweet — Analisis Sentimen WFH ASN

Notebook ini membantu kamu **melabeli tweet secara manual** (positif / netral / negatif) untuk keperluan evaluasi/validasi model.

---
### Panduan label:
- **positif (p)** → mendukung / setuju / senang dengan kebijakan WFH ASN Jumat
- **negatif (n)** → menolak / mengkritik / kecewa dengan kebijakan
- **netral (x)** → hanya menyebarkan info / berita tanpa opini jelas

### Cara pakai:
1. Upload file CSV tweet kamu
2. Jalankan sel labeling → tweet muncul satu per satu
3. Ketik `p` (positif), `n` (negatif), atau `x` (netral), lalu Enter
4. Ketik `q` kalau mau berhenti (progress tersimpan otomatis)
5. Download hasilnya

## 1. Upload file CSV tweet

In [ ]:
import pandas as pd
import os

# OPSI A: Upload dari komputer
from google.colab import files
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
print(f'File dimuat: {INPUT_FILE}')

# OPSI B: Kalau file sudah ada di Colab (uncomment baris ini, comment Opsi A)
# INPUT_FILE = 'tweets-data/wfh_asn.csv'

In [ ]:
import re

df = pd.read_csv(INPUT_FILE)
print(f'Total tweet: {len(df)}')
print('Kolom:', list(df.columns))

# Deteksi kolom teks
TEXT_COLS = ['full_text', 'text', 'tweet', 'content', 'rawContent']
TEXT_COL = next((c for c in TEXT_COLS if c in df.columns), df.columns[-1])
print(f'Kolom teks: {TEXT_COL}')

# Bersihkan teks
def bersihkan(t):
    t = re.sub(r'http\S+|www\.\S+', '', str(t))
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#', '', t)
    return re.sub(r'\s+', ' ', t).strip()

df['clean_text'] = df[TEXT_COL].apply(bersihkan)
df.head()

## 2. Konfigurasi labeling

- `JUMLAH_LABEL` = berapa tweet yang mau kamu labeli (rekomendasi: 300–400).
- `OUTPUT_FILE` = nama file hasil labeling.

In [ ]:
JUMLAH_LABEL = 400   # Jumlah tweet yang akan dilabeli manual
OUTPUT_FILE = 'tweet_berlabel.csv'

# Ambil sample acak (supaya tidak bias urutan)
df_sample = df.sample(n=min(JUMLAH_LABEL, len(df)), random_state=42).reset_index(drop=True)
print(f'Akan melabeli {len(df_sample)} tweet.')

# Kalau ada file hasil labeling sebelumnya (lanjutkan dari terakhir)
if os.path.exists(OUTPUT_FILE):
    df_sudah = pd.read_csv(OUTPUT_FILE)
    mulai_dari = len(df_sudah)
    print(f'Melanjutkan dari tweet ke-{mulai_dari + 1} (sudah ada {mulai_dari} label)')
else:
    df_sudah = pd.DataFrame()
    mulai_dari = 0
    print('Mulai dari awal.')

## 3. Mulai labeling!

- Ketik **`p`** = positif, **`n`** = negatif, **`x`** = netral
- Ketik **`s`** = skip (kalau tweet ambigu / tidak relevan)
- Ketik **`q`** = berhenti & simpan progress

Progress disimpan otomatis setiap 10 tweet.

In [ ]:
LABEL_MAP = {'p': 'positif', 'n': 'negatif', 'x': 'netral', 's': 'skip'}
hasil_label = []

print('='*60)
print('MULAI LABELING (ketik p/n/x/s/q)')
print('='*60)

try:
    for i in range(mulai_dari, len(df_sample)):
        row = df_sample.iloc[i]
        teks = row['clean_text']

        print(f'\n--- Tweet {i+1}/{len(df_sample)} ---')
        print(f'\033[1m{teks}\033[0m')  # Bold
        print()

        while True:
            label = input('[p]ositif / [n]egatif / [x]netral / [s]kip / [q]uit: ').strip().lower()
            if label in LABEL_MAP or label == 'q':
                break
            print('Input tidak valid. Ketik p, n, x, s, atau q.')

        if label == 'q':
            print('\nBerhenti. Menyimpan progress...')
            break

        hasil_label.append({
            'index': i,
            'text': row[TEXT_COL],
            'clean_text': teks,
            'label': LABEL_MAP[label]
        })

        # Auto-save tiap 10 tweet
        if len(hasil_label) % 10 == 0:
            df_baru = pd.concat([df_sudah, pd.DataFrame(hasil_label)], ignore_index=True)
            df_baru.to_csv(OUTPUT_FILE, index=False)
            print(f'  [auto-saved: {len(df_baru)} tweet berlabel]')

except KeyboardInterrupt:
    print('\nDihentikan. Menyimpan progress...')

# Simpan final
df_final = pd.concat([df_sudah, pd.DataFrame(hasil_label)], ignore_index=True)
df_final = df_final[df_final['label'] != 'skip']  # Buang yang di-skip
df_final.to_csv(OUTPUT_FILE, index=False)
print(f'\nTersimpan: {len(df_final)} tweet berlabel di {OUTPUT_FILE}')
print(df_final['label'].value_counts())

## 4. Lihat ringkasan & distribusi label

In [ ]:
import matplotlib.pyplot as plt

df_label = pd.read_csv(OUTPUT_FILE)
print(f'Total tweet berlabel: {len(df_label)}')
print()
print(df_label['label'].value_counts())

ringkasan = df_label['label'].value_counts().reindex(['positif', 'netral', 'negatif'])
ringkasan.plot(kind='bar', color=['#2ecc71', '#95a5a6', '#e74c3c'])
plt.title('Distribusi Label Manual')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Download hasil

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

## 6. (Opsional) Evaluasi model IndoBERT vs label manual

Bandingkan hasil prediksi model dengan label manual kamu → hitung akurasi, precision, recall, F1.

In [ ]:
!pip install -q transformers torch scikit-learn

from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np

MODEL_LABEL_MAP = {'LABEL_0': 'positif', 'LABEL_1': 'netral', 'LABEL_2': 'negatif'}

df_eval = pd.read_csv(OUTPUT_FILE)
df_eval = df_eval[df_eval['label'].isin(['positif', 'netral', 'negatif'])].reset_index(drop=True)

clf = pipeline('sentiment-analysis',
               model='mdhugol/indonesia-bert-sentiment-classification',
               truncation=True, max_length=256)

prediksi = clf(df_eval['clean_text'].tolist())
df_eval['prediksi_model'] = [MODEL_LABEL_MAP.get(r['label'], r['label']) for r in prediksi]

print('\n=== EVALUASI MODEL vs LABEL MANUAL ===')
print(f'Akurasi: {accuracy_score(df_eval["label"], df_eval["prediksi_model"]):.2%}')
print()
print(classification_report(df_eval['label'], df_eval['prediksi_model'],
                            target_names=['positif', 'netral', 'negatif']))

# Confusion Matrix
cm = confusion_matrix(df_eval['label'], df_eval['prediksi_model'],
                      labels=['positif', 'netral', 'negatif'])
print('Confusion Matrix:')
print(pd.DataFrame(cm, index=['Aktual: positif', 'Aktual: netral', 'Aktual: negatif'],
                   columns=['Prediksi: positif', 'Prediksi: netral', 'Prediksi: negatif']))